# Week 1.4: Prompt Engineering for Digital Twins

## Learning Objectives
- Master different prompting techniques
- Understand zero-shot, few-shot, and chain-of-thought
- Apply prompting to digital twin interactions

## Key Insight
> 'Moving from Chat to System Instructions'

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict

print('✅ Ready for prompting exercises!')

✅ Ready for prompting exercises!


In [30]:
import os
from groq import Groq
import os, json
#IMPORTANT: set your real API key securely
# option 1 (recommended): Set this in your environment before running the notebook:
# %env GROQ_API_KEY=sk_...
# Option 2 (for quick local testing only): paste it here (NOT In shared notebooks)
os.environ['GROQ_API_KEY'] = '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'

client= Groq(
    api_key=os.environ.get("GROQ_API_KEY")
)
print("Connected. Ready.")

def llm(prompt: str) -> str:
  """Simple helper for single-turn prompts."""
  chat_completion = client.chat.completions.create(
    model="openai/gpt-oss-120b", # Updated model name to a currently available one
    messages=[
      {"role": "user", "content": prompt}
    ],
  )
  return chat_completion.choices[0].message.content

Connected. Ready.


In [7]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.1 MB/s eta 0:00:00


## Part 1: Prompting Fundamentals

### 1.1 Zero-Shot Prompting

In [32]:
# Zero-shot Prompting
# Zero-shot: Describe the task only
# No examples, no format instructions
# Model infers the answer from training

prompt = '''
Predict the next activity:
I just struggled to get out of bed, because I don't feel too sound, I just got back from some business encounters that I needed to attend to, and now...
'''
response = llm(prompt)
print(response)


...and now I’m probably going to crawl back under the blankets, grab a glass of water, and try to soothe whatever’s making me feel off. I’ll likely take a quick pain reliever or some cold medicine, turn on a calming playlist, and maybe scroll through a few work emails just to make sure nothing urgent slipped through while I was out. After that, I’ll settle in for a short nap or at least a quiet rest before attempting to get back to the rest of my day.


In [23]:
print('Fetching available Groq models...')

try:
    models = client.models.list()
    print("\nCurrently supported Groq models:")
    for model in models.data:
        print(f"- {model.id}")
except Exception as e:
    print(f"Error fetching models: {e}")

Fetching available Groq models...

Currently supported Groq models:
- canopylabs/orpheus-arabic-saudi
- groq/compound
- allam-2-7b
- openai/gpt-oss-120b
- whisper-large-v3-turbo
- openai/gpt-oss-safeguard-20b
- openai/gpt-oss-20b
- meta-llama/llama-prompt-guard-2-22m
- canopylabs/orpheus-v1-english
- whisper-large-v3
- meta-llama/llama-prompt-guard-2-86m
- groq/compound-mini
- qwen/qwen3.8-27b


In [33]:
# Example prompts for digital twin
zero_shot_examples = [
    {
        'task': 'activity_prediction',
        'prompt': 'Predict the next activity: I just finished work, had dinner, and now...',
        'expected': 'leisure/relaxation activity'
    },
    {
        'task': 'preference_extraction',
        'prompt': 'Extract preferences from: I love outdoor activities but hate crowded places',
        'expected': {'likes': ['outdoor', 'quiet'], 'dislikes': ['crowds']}
    }
]

print('Zero-Shot Examples:')
for ex in zero_shot_examples:
    print(f"\nTask: {ex['task']}")
    print(f"Prompt: {ex['prompt']}")
    print(f"Expected: {ex['expected']}")

Zero-Shot Examples:

Task: activity_prediction
Prompt: Predict the next activity: I just finished work, had dinner, and now...
Expected: leisure/relaxation activity

Task: preference_extraction
Prompt: Extract preferences from: I love outdoor activities but hate crowded places
Expected: {'likes': ['outdoor', 'quiet'], 'dislikes': ['crowds']}


In [36]:
# Few-shot: show examples do not explain
#2-3 Input/output pairs before the real questions
# Model learns format from demonstration

Prompt = '''
Examples:
Input: Finished gym, lunch, coffee
Output: {"next":" Work", "confidence":0.82}

Input: Dinner, watched TV
Output: {"next":"Sleep", "confidence":0.91}

Input: Finished work, had dinner, watched TV, now
Output:
'''
response = llm(Prompt)
print(response)

{"next":"Sleep","confidence":0.94}


In [37]:
# 2.1 - CoT vs Plain

# Chain-of-thoughts: four extra words
# Append: Think step by step.
# Model shifts to explicit reasoning

Plain = ''' Should I exercise today?
context: worked 10h, slept 5h.'''

cot = ''' Should I exercise today?
context: worked 10h, slept 5h.
Think step by step.'''

print('---PLAIN---')
print(llm(Plain))
print()
print('--- CHAIN-OF-THOUGHT---')
print(llm(cot))

---PLAIN---
It depends on how you’re feeling right now. A 10‑hour workday and only 5 hours of sleep can leave you depleted, but a brief, low‑intensity session can actually boost energy and help you recover faster—*if* you listen to your body.

### Quick “Decision Tree”

| How you feel after work | Suggested action |
|------------------------|------------------|
| **Very tired, heavy limbs, mental fog** | **Rest or very light movement** (e.g., 5‑10 min stretch, a short walk, gentle yoga). Give your nervous system a break and aim for a proper night’s sleep. |
| **Moderately fatigued but not sore** | **Light to moderate activity** (20‑30 min). Choose something that raises heart rate just enough to get blood flowing without draining you (brisk walk, easy bike ride, body‑weight circuit). |
| **Energetic, no aches, craving movement** | **Full workout** (30‑45 min). You can go for a typical routine—strength, cardio, or a mix—just keep the intensity moderate (RPE 5‑6/10). |

### Why a *light* 

In [38]:
# 2.2 - Build the Twin's system instruction

system = '''
ROLE: You are a digital twin for {user}.
You represent their preferences and
behavioral patterns.

KNOWLEDGE: You have the user's activity
 history and temporal patterns.

 CONSTRAINTS: Alway's return structured output
 never guess. If data is missing,
 say so explicitly.

TONE: Precise, Analytical.
'''

r = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": system},
        {"role": "user", "content": "What next?"}
    ],
)
print(r.choices[0].message.content)

{
  "suggested_next_steps": null,
  "reason": "Insufficient contextual data to determine the appropriate next action.",
  "required_information": [
    "Recent activity or task the user is currently engaged in",
    "Specific goal or domain (e.g., work, personal planning, learning, entertainment)",
    "Time constraints or deadlines, if any"
  ],
  "request_for_clarification": "Please provide details about your current context or objective so I can generate a precise recommendation."
}


In [42]:
# 2.2 - Build the Twin's system Instruction

user = "Chisom"

user_profile = {
    "name": "Chisom",
    "role": "Recent Graduate who just Completed NYSC learning software development",
    "preferences": [
        "prefers precise human explanations",
        "likes structured responses",
        " Works often with Javascript, Python and presentation"
    ],
    "goals": [
        "improve how well she understands the skill she's learning",
        "enhance her life by perfecting her skills and make more money with a paying job",
        "travel around the world and meet people of like minds"
    ]
}

activity_history = [
    {"time": "2026-03-07", "activity": "Attended a two weeks software development bootcamp"},
    {"time": "2026-06-17", "activity": "Deployed and submitted her first website For a knowledge showcase"},
    {"time": "2026-o8-29", " activity": "Rounded up a 3-months scholarship beginner's software development training with 3MTT"}
]
temporal_patterns = {
    "most_active_period": "early mornings",
    "common_tasks": [
        "has a daily routine of showing up on Linkedin professionally",
        "attend online training lectures",
        "searches the Internet for job opportunities that can fetch her money"
    ],
    "recurring_focus": [
        "emerging technologies",
        "good paying job opportunities",
        "fully funded scholarship opportunities"
    ]
}

system = f"""
ROLE:
You are a digital twin for {user}.
You simulate the user's likely priorities, preferences,
and next actions using only the evidence provided.


AVAILABLE DATA:
1. User profile:
{user_profile}

2. Activity history:
{activity_history}

3. Temporal patterns:
{temporal_patterns}


OBJECTIVE:
Given a user query produce informed and structured assessment of:
- the user's likely current context,
- the most probable next actions,
- the evidence supporting each action,
- any missing data that limits confidence.

REASONING RULES:
- Use only the supplied data.
- Do not invent facts.
- Separate observed facts from inferred conclusions.
- If evidence is weak or missing, say so explicitly.
- Rank likely next actions by confidence.
- Ground every inference in profile, history, or temporal patterns.

OUTPUT FORMAT:
Return valid JSON with this structure:
{{
  "user": "string",
  "current_context": {{
    "summary": "string",
    "observed_signals": ["string"]
  }},
  "predicted_next_actions": [
    {{
      "action": "string",
      "confidence": "high | medium | low",
      "why": ["string"],
      "supporting_data": ["string"]
    }}
  ],
  "missing_information": ["string"],
  "response_quality": {{
    "grounded_in_data": true,
    " Contains_guessing": false
  }}
}}

TONE:
Precise, analytical, concise.
"""



In [43]:
r = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": system},
        {"role": "user", "content": "What next?"}
    ],
    response_format={"type": "json_object"}  # use only if supported by your client/model
)

print(r.choices[0].message.content)

{
  "user": "Chisom",
  "current_context": {
    "summary": "Chisom has recently completed a 3‑month beginner software development scholarship, deployed her first website, and regularly engages in early‑morning LinkedIn activity, online training, and job searches focused on emerging technologies and funded opportunities.",
    "observed_signals": [
      "Recent deployment of a showcase website (2026-06-17)",
      "Completion of a 3‑month scholarship (2026-08-29)",
      "Profile preference for precise, structured guidance",
      "Temporal pattern of early‑morning productivity"
    ]
  },
  "predicted_next_actions": [
    {
      "action": "Update LinkedIn profile with the new website project and share a concise post highlighting the achievement",
      "confidence": "high",
      "why": [
        "Daily routine includes professional LinkedIn presence",
        "Recent tangible deliverable (website) provides fresh content",
        "Goal to attract paying job opportunities"
      ],


### 🎯 Exercise 1.1 (Easy): Create Your Prompts

**Task**: Design prompts for your digital twin use case.

In [ ]:
# YOUR CODE HERE
my_prompts = [
    # Add your prompts
]